<a href="https://colab.research.google.com/github/Prajjwal2123/internship-assignments/blob/main/week_7_prajjwal_sharma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install -q sentence-transformers faiss-cpu pypdf gradio
print(" Done")

 Done


In [35]:
import os, re, textwrap, numpy as np
from pathlib import Path
from typing import List, Dict
import torch
import faiss
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print(" Imports done")

 Imports done


In [36]:
from google.colab import files

print("Upload your PDF or TXT file:")
uploaded = files.upload()
uploaded_path = list(uploaded.keys())[0]
print(f" Uploaded: {uploaded_path}")

Upload your PDF or TXT file:


Saving Ensemble-Learning.pdf to Ensemble-Learning (2).pdf
 Uploaded: Ensemble-Learning (2).pdf


In [37]:
def extract_text(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        reader = PdfReader(path)
        pages = []
        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                pages.append(f"[Page {i+1}]\n{text}")
        text = "\n\n".join(pages)
    else:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()

    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text).strip()
    print(f" Extracted {len(text.split()):,} words from {Path(path).name}")
    return text

raw_text = extract_text(uploaded_path)

 Extracted 3,112 words from Ensemble-Learning (2).pdf


In [38]:
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> List[str]:
    words = text.split()
    chunks = []
    step = max(1, chunk_size - overlap)
    i = 0
    while i < len(words):
        chunk = " ".join(words[i : i + chunk_size])
        if len(chunk.strip()) > 20:
            chunks.append(chunk)
        i += step
    return chunks

chunks_raw = chunk_text(raw_text, chunk_size=300, overlap=50)
doc_name = Path(uploaded_path).name

chunks = [
    {"text": c, "source": doc_name, "chunk_id": f"{doc_name}::chunk_{i}"}
    for i, c in enumerate(chunks_raw)
]

print(f"Created {len(chunks)} chunks")

Created 13 chunks


In [39]:
EMBED_MODEL = "all-MiniLM-L6-v2"
print(f"Loading embedding model: {EMBED_MODEL} ...")

embedder = SentenceTransformer(EMBED_MODEL)
texts = [c["text"] for c in chunks]

print(f"Embedding {len(texts)} chunks...")
embeddings = embedder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
).astype(np.float32)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f" FAISS index ready — {index.ntotal} vectors, dim={dim}")

Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding 13 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 FAISS index ready — 13 vectors, dim=384


In [40]:
LLM_MODEL = "google/flan-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {LLM_MODEL} on {device.upper()} ...")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL).to(device)
llm.eval()

print(" LLM ready")

Loading google/flan-t5-base on CPU ...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


 LLM ready


In [41]:
def retrieve(query: str, top_k: int = 5) -> List[Dict]:
    q_emb = embedder.encode(
        [query], normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        chunk = dict(chunks[idx])
        chunk["score"] = float(score)
        results.append(chunk)
    return results


def build_prompt(question: str, context_chunks: List[Dict]) -> str:
    parts = [f"[{i+1}] {c['text']}" for i, c in enumerate(context_chunks)]
    context = "\n\n".join(parts)
    return (
        "Answer the question using only the context below.\n"
        "If the answer is not in the context, say: 'I don't know based on the provided document.'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )


def generate(prompt: str) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)
    with torch.no_grad():
        output = llm.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()


def ask(question: str, top_k: int = 5, verbose: bool = True) -> str:
    retrieved = retrieve(question, top_k=top_k)

    if verbose:
        print(f"🔍 Retrieved {len(retrieved)} chunks:")
        for c in retrieved:
            print(f"   [{c['score']:.3f}] {c['text'][:80]}...")
        print()

    prompt = build_prompt(question, retrieved)
    answer = generate(prompt)
    return answer


print(" RAG pipeline ready")

 RAG pipeline ready


In [44]:
question = "What is ensemble learning?"

answer = ask(question, top_k=5, verbose=True)

print("=" * 60)
print(f"Q: {question}")
print("=" * 60)
print(f"A: {textwrap.fill(answer, width=80)}")
print("=" * 60)

🔍 Retrieved 5 chunks:
   [0.705] a dataset is known as training. • Why is this not called learning? First, note t...
   [0.667] [Page 1] Ensemble Learning Methods Tushar B. Kute, http://tusharkute.com [Page 2...
   [0.644] learning. • Moreover, Ensemble-based models can be incorporated in both of the t...
   [0.637] in a lower variance than any model used in the ensemble. • A voting ensemble is ...
   [0.593] case of regression, and probability values, probability like values, or class la...

Q: What is ensemble learning?
A: Learning


In [43]:
import gradio as gr

def gradio_ask(question, top_k):
    if not question.strip():
        return "Please enter a question.", ""
    retrieved = retrieve(question, top_k=int(top_k))
    prompt = build_prompt(question, retrieved)
    answer = generate(prompt)
    chunks_text = "\n\n".join(
        f"[{i+1}] score={c['score']:.3f}\n{c['text'][:300]}"
        for i, c in enumerate(retrieved)
    )
    return answer, chunks_text

with gr.Blocks(title="RAG Q&A") as demo:
    gr.Markdown(f"## 📄 RAG Q&A — {doc_name}")
    gr.Markdown(f"**{len(chunks)} chunks indexed**")
    question_box = gr.Textbox(label="Your question", placeholder="Ask anything about your document...")
    top_k_slider = gr.Slider(1, 10, value=5, step=1, label="Top-K chunks")
    ask_btn = gr.Button("Ask", variant="primary")
    answer_box = gr.Textbox(label="Answer", lines=5, interactive=False)
    chunks_box = gr.Textbox(label="Retrieved chunks", lines=10, interactive=False)
    ask_btn.click(gradio_ask, inputs=[question_box, top_k_slider], outputs=[answer_box, chunks_box])
    question_box.submit(gradio_ask, inputs=[question_box, top_k_slider], outputs=[answer_box, chunks_box])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://780c22bffe40877335.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
